## tl;dr
The shared pose-only calibration improves the independent train-pair median from 0.545406 to 0.475516 pixels, but the dense canary fails all three native F/J/L gates. Face LPIPS worsens from 0.047989 to 0.059505; the neck patch and tube defects remain. Reject promotion. Original pipeline defaults and the published 50-frame campaign are unchanged.

## Context & Methods
Companion integrity check for the 2026-09-06 pose-only control on clever-shadow. Fit times: 000973, 001059, 001139; held train times: 000979, 001219. One shared 62-camera rig; all original intrinsics fixed. F/J/L RGB is excluded from fitting and prediction. The recorded common post-BA similarity also transforms the held query poses.

### Key Assumptions
Sparse train-pair error is only a preliminary proxy. These held train times were used in earlier diagnostics: they are disjoint from fitting, not an untouched final generalization test. Native render review uses three camera anchors at **one** time, not temporal/fly-through validation. Display face metrics use the same GT-defined study polygon; it excludes most of the neck/hand and differs from the original campaign ROI. No candidate surface mask, semantic mask, or full-frame metric is used.

## Data
Retained local manifests, metrics and native review crops under `/mnt/data/lookcloser_dec5_5a3_surface_repair/pose_only_control`. This notebook checks the completed audit and recomputes paired held errors, rather than rerunning expensive PatchMatch. Requires Python 3, NumPy, pandas and the retained filesystem artifacts.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
root=Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
control=root/'pose_only_control'
workspace=control/'reconstruct_000973'
def read(path): return json.loads(Path(path).read_text())
def digest(path):
    value=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8<<20), b''): value.update(block)
    return value.hexdigest()
findings=read(workspace/'findings.json')
assert findings['audit_status']=='comparison_integrity_pass'
for path,expected in findings['retained_hashes'].items():
    assert digest(path)==expected, path
print('Verified retained hashes:',len(findings['retained_hashes']))
assert findings['depth_maps_verified']==62 and findings['all_intrinsics_identical']
assert findings['original_replay_byte_identical']
assert not findings['temporal_promotion']

Verified retained hashes: 365


## Results
### Independent held-train gate
Pair-block errors are compared on exactly the same ordered camera/time pairs and correspondence counts, without candidate-specific exclusions.

In [2]:
scores=read(control/'held_scores.json')
original='/mnt/data/lookcloser_dec5_5a3_patchmatch_tsdf_50/config/calibration/transforms.json'
candidate=str(control/'calibration.json')
a=scores['results'][original]; b=scores['results'][candidate]
key=lambda row:(row['frame_id'],row['left_camera'],row['right_camera'],row['points'])
assert list(map(key,a))==list(map(key,b))
assert len(a)==len(set(map(key,a)))==716
assert sum(row['points'] for row in a)==153643
assert not set(scores['fit_frames']) & set(scores['held_frames'])
rows=[]
for label,payload in [('original',a),('pose only',b)]:
    values=np.array([r['block_median_absolute_error'] for r in payload])
    assert np.isfinite(values).all()
    rows.append({'calibration':label,'pairs':len(values),'median_px':np.median(values),'p90_px':np.quantile(values,.9)})
display(pd.DataFrame(rows))
selection=read(control/'selection_before_render.json')
assert selection['eligible_for_dense_control']
assert not selection['selection_uses_eval_rgb'] and not selection['selection_uses_render_metrics']
display(pd.DataFrame(selection['per_time']))

,calibration,pairs,median_px,p90_px
0,original,716,0.545406,1.009393
1,pose only,716,0.475516,0.811580


,frame_id,median_before_after,p90_before_after,pairs
0,000979,"[0.5555786845486785, 0.48501590465251343]","[1.0246960370410254, 0.8570289801027428]",358
1,001219,"[0.5373493119233073, 0.4625760591115493]","[0.9707080293687353, 0.7796606775415855]",358


### Paired face metrics and native verdicts
LPIPS is lower-is-better. Face metrics do not replace review of the neck, hand or lipstick outside the face ROI. All crops are native-scale side-by-side comparisons; missing room alone is ignored.

In [3]:
metric_keys=['face_psnr','face_ssim','face_lpips']
base=read(root/'depth_separated_source_control/off_rank001/face_metrics/metrics.json')
new=read(workspace/'face_metrics/metrics.json')
for key in ['ground_truth_sha256','face_polygons_sha256','face_bbox_xyxy','protocol','metric_domain']:
    assert base[key]==new[key], key
rows=[]
for name,m in [('Original rig',base),('Pose only',new)]:
    assert np.isfinite([m[k] for k in metric_keys]).all()
    assert not any(k.startswith(('loss','full_frame','actor_psnr','room_psnr')) for k in m)
    rows.append({'variant':name,**{k:m[k] for k in metric_keys}})
display(pd.DataFrame(rows))
review=read(workspace/'visual_review.json')
assert review['temporal_frame_count']==1
assert len(review['rows'])==3
assert all(row['visual_status']!='pending' for row in review['rows'])
display(pd.DataFrame([{k:r[k] for k in ['anchor','visual_status','visual_notes']} for r in review['rows']]))
print('Native review counts:',findings['visual_counts'])
print('Coverage mean/min:',findings['depth_coverage_mean'],findings['depth_coverage_min'])
print('Mesh vertices/triangles/components:',findings['mesh_vertices'],findings['mesh_triangles'],findings['mesh_components'])

,variant,face_psnr,face_ssim,face_lpips
0,Original rig,28.724285,0.889098,0.047989
1,Pose only,27.008524,0.822349,0.059505


,anchor,visual_status,visual_notes
0,F,fail,The conspicuous warm neck patch and its jagged...
1,J,fail,The tube retains an incorrect broad pale side/...
2,L,fail,"Ear and face remain broadly coherent, but hair..."


Native review counts: {'fail': 3, 'pass': 0, 'uncertain': 0}
Coverage mean/min: 0.385430932024094 0.24966483410493828
Mesh vertices/triangles/components: 81452 157427 1


## Takeaways
A sparse matching improvement is not sufficient for surface acceptance. The recorded native gate, with unchanged original-rig replay hashes and matched face ROI, is the quality evidence. No claim of every-40th-frame or continuous fly-through success is made. See `../dec5_patchmatch_surface_repair.md` for visual interpretation and links to native crops.